# Notebook 2 of 4: One-Time Setup — Baseline Data + Schema Snapshot

Run this **once** before the demo. It:
1. Generates clean baseline data (customers, accounts, transactions)
2. Writes them to lakehouse Delta tables
3. Captures the EXPECTED schema for each entity into the schema snapshot table

After this completes, the Fabric pipeline `SchemaDriftDemo` can be triggered.

In [ ]:
%run ./01_config

In [ ]:
from pyspark.sql import SparkSession
from src.entity_generators import (
    generate_customers, generate_accounts, generate_transactions,
    save_schema_snapshots_delta,
)
from src.schema_drift_config import LH_TABLE

spark = SparkSession.builder.getOrCreate()

print('=' * 72)
print('  ONE-TIME SETUP: BASELINE DATA + SCHEMA SNAPSHOT')
print('=' * 72)

# 1. Generate clean baseline
print('\nGenerating baseline data...')
customer_df = generate_customers(spark)
account_df  = generate_accounts(spark, customer_df)
txn_df      = generate_transactions(spark, account_df)

# 2. Write to Lakehouse Delta tables
print('\nWriting baseline as Delta tables...')
customer_df.write.format('delta').mode('overwrite').saveAsTable(LH_TABLE['raw_customers'])
account_df.write.format('delta').mode('overwrite').saveAsTable(LH_TABLE['raw_accounts'])
txn_df.write.format('delta').mode('overwrite').saveAsTable(LH_TABLE['raw_transactions'])
print(f'  {LH_TABLE["raw_customers"]}')
print(f'  {LH_TABLE["raw_accounts"]}')
print(f'  {LH_TABLE["raw_transactions"]}')

# 3. Snapshot the expected schemas
print('\nSnapshotting expected schemas...')
save_schema_snapshots_delta(spark, LH_TABLE['schema_snapshot'])

print('\n' + '=' * 72)
print('  SETUP COMPLETE — ready for the schema drift demo')
print('=' * 72)
print('\nNext: trigger the Fabric pipeline with one of:')
print('  drift_type = column_added')
print('  drift_type = column_removed   (set force_critical=True for CRITICAL)')
print('  drift_type = type_changed     (set force_critical=True for CRITICAL)')